# BFS preprocess sweep benchmark (Google Colab CPU)

Benchmarks the PAMR path-preprocessing worker (`pmr/models/pr/preprocess.cpp`) while
sweeping its BFS hyperparameters — `max_hops`, `max_fanout`, `decay_factor` — over the
**validation split** of WikiArt-MKG-v2, and records **wall time, CPU time, and RAM**
per configuration into `assets/bfs.csv` on Google Drive.

**Runtime**: ~1.5–2 h on a standard Colab CPU runtime (2 vCPU / ~12.7 GB RAM).

**How to run**
1. Upload nothing manually: this notebook copies the repo and the triples CSV from your Google Drive.
2. `Runtime → Run all`. Cells are sequential and each asserts its own sanity checks.
3. The final CSV lands at `/content/bfs.csv` and is copied to
   `<Drive>/Study/HCMUS/Grad/Doctor/Thesis/Code/pmr/mmkg-pmr/assets/bfs.csv`.

**What is measured per sweep row**
- `wall_time_s`: full worker run (graph load + BFS search, 2 parallel workers).
- `load_wall_s`: a calibration run with a header-only queries file — the constant
  graph-parse cost. `search_time_s = wall_time_s − load_wall_s` approximates the
  BFS-search cost attributable to the swept hyperparameters.
- `peak_concurrent_rss_mb`: sum of RSS across the worker process tree, sampled every 0.25 s.
- `max_worker_rss_mb`: largest single worker RSS observed.
- `found / missing / self_est / avg_hops`: path-quality counters (missing includes
  self-probes; `self_est` re-identifies them via head == tail).
- Determinism spot-check: the production config `(5, 100, 0.5)` is run twice and the
  merged outputs must be byte-identical.

## 1 · Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 2 · Benchmark configuration

In [2]:
from pathlib import Path

REPO_GDRIVE_REL = "Study/HCMUS/Grad/Doctor/Thesis/Code/pmr/mmkg-pmr"
DATA_GDRIVE_REL = "Study/HCMUS/Grad/Doctor/Thesis/Code/pmr/data/wikiart_mkg_v2"
TRIPLES_NAME = "wikiart_mkg_v2_triples.csv"

DRIVE_ROOT = Path("/content/drive/MyDrive")
REPO_GDRIVE = DRIVE_ROOT / REPO_GDRIVE_REL
TRIPLES_GDRIVE = DRIVE_ROOT / DATA_GDRIVE_REL / TRIPLES_NAME

REPO_DIR = Path("/content/mmkg-pmr")
DATA_DIR = Path("/content/data/wikiart_mkg_v2")
TRIPLES_CSV = DATA_DIR / TRIPLES_NAME

WORK_DIR = Path("/content/bfs_bench")
LOG_DIR = WORK_DIR / "worker_logs"
OUT_CSV = Path("/content/bfs.csv")
OUT_CSV_DRIVE = DRIVE_ROOT / REPO_GDRIVE_REL / "assets" / "bfs.csv"

# Working dirs must exist before any log/output file is opened
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

NUM_THREADS = 2                 # Colab CPU runtime has 2 vCPUs
MASK_QUERY_EDGES = True         # production policy: own query edge masked per probe
ALLOW_TARGET_TRAVERSAL = True   # uniform rule: non-own target edges traversable

# (max_hops, max_fanout, decay_factor)
SWEEP = [(h, f, d) for h in (3, 5) for f in (10, 50, 100) for d in (0.5, 1.0)]
DETERMINISM_CONFIG = (5, 100, 0.5)   # production default, run twice

assert REPO_GDRIVE.exists(), f"Repo not found on Drive: {REPO_GDRIVE}"
assert TRIPLES_GDRIVE.exists(), f"Triples CSV not found on Drive: {TRIPLES_GDRIVE}"
print(f"{len(SWEEP)} sweep configs; threads={NUM_THREADS}; mask={MASK_QUERY_EDGES}; "
      f"allow_target={ALLOW_TARGET_TRAVERSAL}")
print(f"repo: {REPO_GDRIVE}\ndata: {TRIPLES_GDRIVE}")

12 sweep configs; threads=2; mask=True; allow_target=True
repo: /content/drive/MyDrive/Study/HCMUS/Grad/Doctor/Thesis/Code/pmr/mmkg-pmr
data: /content/drive/MyDrive/Study/HCMUS/Grad/Doctor/Thesis/Code/pmr/data/wikiart_mkg_v2/wikiart_mkg_v2_triples.csv


## 3 · Copy repo + triples CSV from Drive to /content
Copies the code onto the fast local disk (running from Drive's FUSE mount is slow).
Feature pickles and checkpoints are **not** needed for the BFS benchmark.

In [5]:
import shutil, time
t0 = time.time()
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(REPO_GDRIVE, REPO_DIR,
                ignore=shutil.ignore_patterns("checkpoints", ".git*", "__pycache__", "*.pyc", "*.gsheet"))
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not TRIPLES_CSV.exists():
    print("copying triples CSV (~1 GB, a few minutes) ...")
    shutil.copyfile(TRIPLES_GDRIVE, TRIPLES_CSV)
print(f"ready in {time.time() - t0:.0f}s  |  triples size = "
      f"{TRIPLES_CSV.stat().st_size / 1e9:.2f} GB")
assert (REPO_DIR / "pmr/models/pr/preprocess.cpp").exists()
assert (REPO_DIR / "pmr/scripts/prepare.py").exists()

copying triples CSV (~1 GB, a few minutes) ...
ready in 28s  |  triples size = 1.04 GB


## 4 · Dependencies
Only what the preprocessing path needs. `torch`/`numpy`/`pandas` are preinstalled on
Colab; `salesforce-lavis` and the pinned versions in `requirements.txt` are deliberately
not installed (not used by the BFS pipeline, and pinning torch would downgrade Colab's).

In [6]:
import importlib, subprocess, sys

REQUIRED = [("yaml", "pyyaml"), ("tqdm", "tqdm"), ("psutil", "psutil"),
            ("pandas", "pandas"), ("numpy", "numpy"), ("torch", "torch")]
for module, package in REQUIRED:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import numpy, pandas, torch, yaml, tqdm, psutil
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| torch", torch.__version__,
      "| psutil", psutil.__version__)

numpy 2.1.3 | pandas 2.2.3 | torch 2.11.0+cpu | psutil 5.9.5


## 5 · Compile the C++ worker
Tries `-std=c++23`, then falls back to `c++20` / `c++17` (no C++23-specific features are used).

In [7]:
import subprocess

SRC = REPO_DIR / "pmr" / "models" / "pr" / "preprocess.cpp"
BIN = REPO_DIR / "pmr" / "models" / "pr" / "preprocess.o"

compiled_with = None
for std in ("c++23", "c++20", "c++17"):
    cmd = ["g++", f"-std={std}", "-O3", "-pthread", "-o", str(BIN), str(SRC)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        compiled_with = std
        break
    print(f"-std={std} failed:\n{r.stderr[-400:]}")
assert compiled_with, "g++ compilation failed for every standard"
print(f"compiled with -std={compiled_with} -> {BIN}")

compiled with -std=c++23 -> /content/mmkg-pmr/pmr/models/pr/preprocess.o


## 6 · Derived benchmark configs
The production YAMLs carry absolute local (`/Users/...`) paths, so the notebook derives
benchmark copies with `/content` paths, `keep_temp: true`, the val-only split, and the
production masking policy. Everything else (registry pairs, relation filter, negatives
per query) is copied verbatim from the production configs.

In [8]:
import json, yaml

BENCH_DIR = Path("/content/bench")
PREP_DIR = BENCH_DIR / "prep"
PREPROC_DIR = BENCH_DIR / "preproc"

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def dump_yaml(cfg, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

base_prepare = load_yaml(REPO_DIR / "pmr/configs/eval/wikiart_mkg_v2_neg_sampling.yaml")
bench_prepare = dict(base_prepare)
bench_prepare["data_path"] = str(TRIPLES_CSV)
bench_prepare["output_dir"] = str(PREP_DIR)
BENCH_PREPARE_CFG = REPO_DIR / "pmr/configs/eval/bench_neg_sampling.yaml"
dump_yaml(bench_prepare, BENCH_PREPARE_CFG)

base_paths = load_yaml(REPO_DIR / "pmr/configs/pr/wikiart_mkg_v2_paths.yaml")
bench_paths = dict(base_paths)
bench_paths["graph_csv"] = str(TRIPLES_CSV)
bench_paths["prepare_config"] = "configs/eval/bench_neg_sampling.yaml"
bench_paths["output_dir"] = str(PREPROC_DIR)
bench_paths["num_threads"] = NUM_THREADS
bench_paths["keep_temp"] = True
bench_paths["path_graph"] = {
    "mask_query_edges": MASK_QUERY_EDGES,
    "allow_target_relation_traversal": ALLOW_TARGET_TRAVERSAL,
}
BENCH_PATHS_CFG = REPO_DIR / "pmr/configs/pr/bench_paths_val.yaml"
dump_yaml(bench_paths, BENCH_PATHS_CFG)

print("derived prepare config:", BENCH_PREPARE_CFG)
print("derived paths config:  ", BENCH_PATHS_CFG)
print(json.dumps({k: bench_paths[k] for k in
                  ("output_dir", "num_threads", "keep_temp", "path_graph")}, indent=2))

derived prepare config: /content/mmkg-pmr/pmr/configs/eval/bench_neg_sampling.yaml
derived paths config:   /content/mmkg-pmr/pmr/configs/pr/bench_paths_val.yaml
{
  "output_dir": "/content/bench/preproc",
  "num_threads": 2,
  "keep_temp": true,
  "path_graph": {
    "mask_query_edges": true,
    "allow_target_relation_traversal": true
  }
}


## 7 · Generate queries (`prepare.py`)
Uses the new 40-ID registry. The four target relations are small (~815 train / ~176 val /
~206 test triples), so this step is dominated by the one-time DataLoader construction
over the 8.29M-row CSV (~5–15 min, several GB RAM — run as a subprocess).

In [9]:
import subprocess, sys, time, json

PREP_LOG = WORK_DIR / "prepare.log"
t0 = time.time()
with open(PREP_LOG, "w") as log:
    r = subprocess.run(
        [sys.executable, "-m", "pmr.scripts.prepare",
         "--config", "configs/eval/bench_neg_sampling.yaml"],
        cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
print(f"prepare.py rc={r.returncode} in {time.time() - t0:.0f}s  (log: {PREP_LOG})")
print("\n".join(open(PREP_LOG).read().splitlines()[-12:]))
assert r.returncode == 0, "prepare.py failed — see log"

PREP_DIR = BENCH_DIR / "prep"
VAL_QUERIES_RAW = PREP_DIR / "val_queries.csv"
assert VAL_QUERIES_RAW.exists()
meta = json.load(open(PREP_DIR / "metadata.json"))
val_meta = meta["splits"]["val"]
print("val split metadata:", json.dumps(val_meta, indent=2))
assert val_meta["negatives_full_pct"] >= 99.0, "val negatives incomplete"
print("prepare OK")

FileNotFoundError: [Errno 2] No such file or directory: '/content/bfs_bench/prepare.log'

## 8 · Build the val-split BFS fixture (`graph_val.csv` + BFS queries)
Runs the production preprocessor for the **val split only** (`--splits val`,
`--keep-temp`): derives entity types, writes the train-scoped graph (~8M rows,
a few minutes) and the 6-column BFS queries CSV (~1,936 probes). This run also
executes the default BFS `(5, 100, 0.5)` uninstrumented — the sweep below re-runs
it instrumented.

In [ ]:
import subprocess, sys, time
import pandas as pd

PREPROC_LOG = WORK_DIR / "preprocess.log"
t0 = time.time()
with open(PREPROC_LOG, "w") as log:
    r = subprocess.run(
        [sys.executable, "-m", "pmr.models.pr.preprocess",
         "--config", "configs/pr/bench_paths_val.yaml",
         "--no-compile", "--splits", "val"],
        cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
print(f"preprocess.py rc={r.returncode} in {time.time() - t0:.0f}s  (log: {PREPROC_LOG})")
print("\n".join(open(PREPROC_LOG).read().splitlines()[-14:]))
assert r.returncode == 0, "preprocess.py failed — see log"

run_dirs = sorted(PREPROC_DIR.glob("*"), key=lambda p: p.stat().st_mtime)
RUN_DIR = run_dirs[-1]
GRAPH_VAL = RUN_DIR / "graph_val.csv"
QUERIES_VAL = RUN_DIR / "val_queries.csv"
assert GRAPH_VAL.exists() and QUERIES_VAL.exists(), (RUN_DIR, list(RUN_DIR.iterdir()))

VAL_QDF = pd.read_csv(QUERIES_VAL)
assert list(VAL_QDF.columns) == [
    "row_index", "query_id", "head_id", "tail_id",
    "query_relation_id", "inverse_query_relation_id"], VAL_QDF.columns
BENCH_ROW_META = {
    int(r.row_index): {"head_id": int(r.head_id), "tail_id": int(r.tail_id)}
    for r in VAL_QDF.itertuples(index=False)
}
NUM_PROBES = len(VAL_QDF)
graph_rows = sum(1 for _ in open(GRAPH_VAL)) - 1  # minus header
print(f"run_dir={RUN_DIR}")
print(f"graph_val rows: {graph_rows:,}  |  probes: {NUM_PROBES:,}  "
      f"({VAL_QDF['query_id'].nunique():,} queries)")
assert NUM_PROBES > 1000, "suspiciously few probes — check the prepare step"

# Target relation IDs for the worker's target-traversal argument
manifest = json.load(open(RUN_DIR / "relation_manifest.json"))
TARGET_NAMES = base_prepare.get("relation_filter", {}).get("relations", [])
TARGET_IDS_CSV = ",".join(str(manifest["relation_to_id"][n]) for n in TARGET_NAMES)
print("target relation ids:", TARGET_IDS_CSV)
print("fixtures ready")

## 9 · Instrumentation helpers (time + RAM)

In [ ]:
import subprocess, threading, time, os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import psutil

LOG_DIR.mkdir(parents=True, exist_ok=True)

def _run_worker(args_tuple):
    (binary, graph, queries, out_prefix, hops, fanout, decay,
     tid, threads, log_dir, mask, allow, targets) = args_tuple
    cmd = [str(binary), str(graph), str(queries), str(out_prefix),
           str(hops), str(fanout), str(decay),
           str(tid), str(threads), str(log_dir),
           str(int(mask)), str(int(allow)), targets]
    t0 = time.perf_counter()
    r = subprocess.run(cmd, capture_output=True, text=True)
    return {"tid": tid, "rc": r.returncode,
            "stderr": (r.stderr or "")[-800:] if r.returncode else "",
            "wall_s": time.perf_counter() - t0}

def run_bfs_once(graph, queries, out_prefix, hops, fanout, decay,
                 threads=NUM_THREADS):
    """Spawn `threads` C++ workers exactly like production and measure."""
    peak_concurrent = [0.0]
    max_worker = [0.0]
    stop = threading.Event()

    def sampler():
        me = psutil.Process(os.getpid())
        while not stop.is_set():
            total, mx = 0, 0
            for p in me.children(recursive=True):
                try:
                    rss = p.memory_info().rss
                    total += rss
                    mx = max(mx, rss)
                except psutil.Error:
                    pass
            peak_concurrent[0] = max(peak_concurrent[0], total)
            max_worker[0] = max(max_worker[0], mx)
            stop.wait(0.25)

    cpu0 = os.times()
    t0 = time.perf_counter()
    sampler_thread = threading.Thread(target=sampler)
    sampler_thread.start()
    tasks = [(BIN, graph, queries, str(out_prefix), hops, fanout, decay,
              tid, threads, LOG_DIR, MASK_QUERY_EDGES,
              ALLOW_TARGET_TRAVERSAL, TARGET_IDS_CSV)
             for tid in range(threads)]
    with ThreadPoolExecutor(max_workers=threads) as ex:
        results = list(ex.map(_run_worker, tasks))
    stop.set()
    sampler_thread.join()
    wall = time.perf_counter() - t0
    cpu1 = os.times()
    cpu_s = ((cpu1.children_user - cpu0.children_user)
             + (cpu1.children_system - cpu0.children_system))
    worker_ok = all(r["rc"] == 0 for r in results)
    if not worker_ok:
        print("WORKER FAILURES:", [r for r in results if r["rc"] != 0])
    return {"wall_s": wall, "children_cpu_s": cpu_s,
            "peak_concurrent_rss_mb": peak_concurrent[0] / 2 ** 20,
            "max_worker_rss_mb": max_worker[0] / 2 ** 20,
            "worker_ok": worker_ok, "workers": results}

def summarize_outputs(out_prefix, threads):
    """Count found/missing/self probes and audit output completeness."""
    hops_vals, row_indexes, total_bytes = [], [], 0
    for tid in range(threads):
        part = Path(f"{out_prefix}_paths_{tid}.txt")
        if not part.exists():
            continue
        data = part.read_text()
        total_bytes += len(data.encode("utf-8"))
        for line in data.splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split(";")
            assert len(parts) == 7, f"bad worker row: {line[:100]}"
            row_indexes.append(int(parts[1]))
            hops_vals.append(int(parts[2]))
    absent = sorted(set(BENCH_ROW_META) - set(row_indexes))
    duplicated = len(row_indexes) - len(set(row_indexes))
    completeness_ok = (not absent) and duplicated == 0
    found = [h for h in hops_vals if h >= 0]
    missing = [h for h in hops_vals if h == -1]
    self_est = 0
    for h, ri in zip(hops_vals, row_indexes):
        meta = BENCH_ROW_META.get(ri)
        if h == -1 and meta and meta["head_id"] == meta["tail_id"]:
            self_est += 1
    return {"found": len(found), "missing": len(missing),
            "self_est": self_est,
            "avg_hops": (sum(found) / len(found)) if found else 0.0,
            "total_bytes": total_bytes, "completeness_ok": completeness_ok}

def merged_sorted_lines(out_prefix, threads):
    lines = []
    for tid in range(threads):
        part = Path(f"{out_prefix}_paths_{tid}.txt")
        if part.exists():
            lines.extend(part.read_text().splitlines())
    return sorted(line for line in lines if line.strip())

print("instrumentation ready")

## 10 · Calibration: graph-load-only run
An empty queries file isolates the constant cost of parsing `graph_val.csv` and
building the adjacency in each worker. Subtracting `load_wall_s` from a config's
wall time approximates its BFS-search cost.

In [ ]:
import pandas as pd

EMPTY_QUERIES = WORK_DIR / "empty_queries.csv"
pd.read_csv(QUERIES_VAL, nrows=0).to_csv(EMPTY_QUERIES, index=False)

cal = run_bfs_once(GRAPH_VAL, EMPTY_QUERIES, WORK_DIR / "calib", 5, 100, 0.5)
LOAD_WALL = cal["wall_s"]
print(f"load-only calibration: wall={LOAD_WALL:.1f}s  "
      f"peak_rss={cal['peak_concurrent_rss_mb']:.0f}MB  "
      f"max_worker_rss={cal['max_worker_rss_mb']:.0f}MB")
assert cal["worker_ok"]

## 11 · Sweep (12 configs × val probes)
Each config spawns 2 workers on the same graph+queries fixture, measures time/RAM,
merges the outputs, audits probe completeness, and deletes the outputs.

In [ ]:
import datetime
import pandas as pd

records = []

def sweep_row(hops, fanout, decay):
    out_prefix = WORK_DIR / f"out_h{hops}_f{fanout}_d{decay}"
    res = run_bfs_once(GRAPH_VAL, QUERIES_VAL, out_prefix, hops, fanout, decay)
    summary = summarize_outputs(out_prefix, NUM_THREADS)
    for tid in range(NUM_THREADS):
        part = Path(f"{out_prefix}_paths_{tid}.txt")
        if part.exists():
            part.unlink()
    row = {
        "split": "val",
        "max_hops": hops, "max_fanout": fanout, "decay_factor": decay,
        "num_threads": NUM_THREADS,
        "mask_query_edges": MASK_QUERY_EDGES,
        "allow_target_traversal": ALLOW_TARGET_TRAVERSAL,
        "num_probes": NUM_PROBES,
        "worker_ok": res["worker_ok"],
        "completeness_ok": summary["completeness_ok"],
        "wall_time_s": round(res["wall_s"], 3),
        "load_wall_s": round(LOAD_WALL, 3),
        "search_time_s": round(res["wall_s"] - LOAD_WALL, 3),
        "children_cpu_s": round(res["children_cpu_s"], 3),
        "peak_concurrent_rss_mb": round(res["peak_concurrent_rss_mb"], 1),
        "max_worker_rss_mb": round(res["max_worker_rss_mb"], 1),
        "found": summary["found"], "missing": summary["missing"],
        "self_est": summary["self_est"], "avg_hops": round(summary["avg_hops"], 4),
        "output_bytes": summary["total_bytes"],
    }
    records.append(row)
    print(f"[{len(records):2d}/{len(SWEEP)}] h={hops} f={fanout} d={decay}  "
          f"wall={row['wall_time_s']:.1f}s (search≈{row['search_time_s']:.1f}s)  "
          f"peakRSS={row['peak_concurrent_rss_mb']:.0f}MB  "
          f"found={row['found']} missing={row['missing']}")
    assert row["worker_ok"], f"worker failed for config {(hops, fanout, decay)}"
    assert row["completeness_ok"], f"missing probes for config {(hops, fanout, decay)}"
    return row

for hops, fanout, decay in SWEEP:
    sweep_row(hops, fanout, decay)
print("sweep done")

### Determinism spot-check — production config run twice

In [ ]:
det_prefix_a = WORK_DIR / "det_a"
det_prefix_b = WORK_DIR / "det_b"
hops, fanout, decay = DETERMINISM_CONFIG
res_a = run_bfs_once(GRAPH_VAL, QUERIES_VAL, det_prefix_a, hops, fanout, decay)
res_b = run_bfs_once(GRAPH_VAL, QUERIES_VAL, det_prefix_b, hops, fanout, decay)
lines_a = merged_sorted_lines(det_prefix_a, NUM_THREADS)
lines_b = merged_sorted_lines(det_prefix_b, NUM_THREADS)
det_summary = summarize_outputs(det_prefix_a, NUM_THREADS)
for prefix in (det_prefix_a, det_prefix_b):
    for tid in range(NUM_THREADS):
        part = Path(f"{prefix}_paths_{tid}.txt")
        if part.exists():
            part.unlink()
assert res_a["worker_ok"] and res_b["worker_ok"]
assert lines_a == lines_b, "deterministic worker violated: outputs differ"
assert det_summary["completeness_ok"], det_summary
assert len(lines_a) == NUM_PROBES, (len(lines_a), NUM_PROBES)
print(f"determinism OK: {len(lines_a):,} rows identical across runs "
      f"(hops={hops}, fanout={fanout}, decay={decay})")

## 12 · Export CSV → `/content/bfs.csv` → Drive `assets/bfs.csv`

In [ ]:
import shutil
import datetime

df = pd.DataFrame(records)
df.insert(0, "timestamp", datetime.datetime.now().isoformat(timespec="seconds"))
COLUMNS = ["timestamp", "split", "max_hops", "max_fanout", "decay_factor",
           "num_threads", "mask_query_edges", "allow_target_traversal",
           "num_probes", "worker_ok", "completeness_ok",
           "wall_time_s", "load_wall_s", "search_time_s", "children_cpu_s",
           "peak_concurrent_rss_mb", "max_worker_rss_mb",
           "found", "missing", "self_est", "avg_hops", "output_bytes"]
df = df[COLUMNS]
df.to_csv(OUT_CSV, index=False)
OUT_CSV_DRIVE.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(OUT_CSV, OUT_CSV_DRIVE)

print(f"CSV -> {OUT_CSV}")
print(f"CSV -> {OUT_CSV_DRIVE}")
display_cols = ["max_hops", "max_fanout", "decay_factor", "wall_time_s",
                "search_time_s", "peak_concurrent_rss_mb", "max_worker_rss_mb",
                "found", "missing", "avg_hops"]
print(df[display_cols].to_string(index=False))

## 13 · Reading the results
- `search_time_s = wall_time_s − load_wall_s` — the graph-parse cost is constant
  across configs, so this column isolates the effect of `max_hops / max_fanout /
  decay_factor` on BFS search.
- `peak_concurrent_rss_mb` ≈ 2 × `max_worker_rss_mb` (two full adjacency copies) —
  the thread-RAM budget documented in the paths config.
- `found` / `missing`: higher `max_fanout` or `decay_factor` should raise `found`
  (more reachable candidates → more path evidence for PAMR) at some time cost.
- `avg_hops` shows whether relaxed fanout/decay actually yields shorter or longer
  evidence, which affects PAMR's max-token budget (`max_tokens = 2·(hops+1)+1`).